# Background Removal + Canny Edge Detection

This notebook covers the first part of the plate segmentation pipeline:

1. Remove the background and isolate the main object.
2. Keep only the largest valid foreground region.
3. Apply Canny edge detection to the segmented object image.

> Update the `IMAGE_PATH` below to point to any image in your dataset.

In [ ]:
import cv2
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
IMAGE_PATH = "/data/seg/test/good"

try:
    from rembg import remove, new_session
    REMBG_AVAILABLE = True
except Exception:
    REMBG_AVAILABLE = False

# Pick the first available image from the dataset folder.
# You can also set this to a single file path, for example:
# IMAGE_PATH = Path("data/seg/test/good/image_001.jpg")
image_dir = Path("data/seg/test/good")
image_candidates = sorted(image_dir.glob("*")) if image_dir.exists() else []
IMAGE_PATH = image_candidates[0] if image_candidates else None

if IMAGE_PATH is None:
    raise FileNotFoundError("No image found in data/seg/test/good. Update IMAGE_PATH manually.")

print(f"Using image: {IMAGE_PATH}")

In [ ]:
def load_image(path):
    img = Image.open(path).convert("RGB")
    return np.array(img)


def remove_background(img):
    """Returns a binary mask of the main object in the image."""

    if REMBG_AVAILABLE:
        session = new_session("u2net")
        raw_mask = np.array(remove(Image.fromarray(img), session=session, only_mask=True))
        mask = (raw_mask > 127).astype(np.uint8)
    else:
        h, w = img.shape[:2]
        scale = 640.0 / max(h, w)
        if scale < 1:
            resized = cv2.resize(img, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
        else:
            resized = img

        resized_h, resized_w = resized.shape[:2]
        mask = np.zeros((resized_h, resized_w), dtype=np.uint8)
        margin_x = int(resized_w * 0.05)
        margin_y = int(resized_h * 0.05)
        rect = (margin_x, margin_y, resized_w - 2 * margin_x, resized_h - 2 * margin_y)

        bgd_model = np.zeros((1, 65), np.float64)
        fgd_model = np.zeros((1, 65), np.float64)
        bgr_img = cv2.cvtColor(resized, cv2.COLOR_RGB2BGR)
        cv2.grabCut(bgr_img, mask, rect, bgd_model, fgd_model, 5, cv2.GC_INIT_WITH_RECT)
        mask = np.where((mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD), 255, 0).astype(np.uint8)

        if scale < 1:
            mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)

    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num_labels > 1:
        largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        mask = (labels == largest_label).astype(np.uint8) * 255

    return mask


def get_segmented_object(img, mask):
    segmented = img.copy()
    segmented[mask == 0] = 0
    return segmented


img = load_image(IMAGE_PATH)
mask = remove_background(img)
segmented = get_segmented_object(img, mask)

print("Mask shape:", mask.shape)
print("Image shape:", img.shape)
print("Background removed object ready.")

## Visualize the Result

The next block shows the original image, the extracted object mask, and the background-free object image.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img)
axes[0].set_title("Original Image")
axes[0].axis("off")

axes[1].imshow(mask, cmap="gray")
axes[1].set_title("Object Mask")
axes[1].axis("off")

axes[2].imshow(segmented)
axes[2].set_title("Background Removed Object")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## Canny Edge Detection on the Segmented Object

After the background is removed, we apply Canny edge detection to the segmented object image to detect object boundaries and outline features.

In [ ]:
# Convert the segmented object image to grayscale for Canny
gray = cv2.cvtColor(segmented, cv2.COLOR_RGB2GRAY)

# Optional smoothing to reduce noise before edge detection
blurred = cv2.GaussianBlur(gray, (5, 5), 0)

# Apply Canny edge detection
edges = cv2.Canny(blurred, threshold1=50, threshold2=150)

# Convert edges back to RGB for display
edges_rgb = cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(segmented)
axes[0].set_title("Segmented Object")
axes[0].axis("off")

axes[1].imshow(blurred, cmap="gray")
axes[1].set_title("Smoothed Grayscale")
axes[1].axis("off")

axes[2].imshow(edges_rgb)
axes[2].set_title("Canny Edges")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## Notes

- If `rembg` is installed, the notebook uses `u2net` for stronger background removal.
- If it is not installed, the code automatically falls back to OpenCV `GrabCut`.
- Tune the Canny thresholds (`threshold1`, `threshold2`) if the edges are too weak or too noisy.